In [5]:
import pandas as pd
import sqlite3
import numpy as np
import os
import platform 
import torch
import pickle
from transformers import GPT2Tokenizer, AutoTokenizer
from torch.nn import functional as F
import sys
from tqdm import tqdm
tqdm.pandas()

In [6]:
DUNDEE_DATA_M_PATH = r'/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/dundee/dundee_df_preloaded.csv'

dundee_df = pd.read_csv(DUNDEE_DATA_M_PATH)
dundee_df["row_UID"] = dundee_df.index
dundee_df["Word"] = dundee_df["Word"].astype(str)
dundee_df

,Word,Text_Nr,Trial,LINE,LINE_WPOS,SERIAL_SCRNUM,INLET_POSLINE,OLEN,Word_Length,PUNCTCODE,...,parafoveal_entropy,parafoveal_surprise,parafoveal_entropy_contextual,parafoveal_surprise_contextual,BAD,word_land_pos,word_land_frac,word_land_dist,word_land_dist_abs,row_UID
0,Are,1,1,1,1,1,0,3,3,0,...,6.738573,5.007107,8.761518,5.755655,False,0.5,0.166667,0.333333,1.0,0
1,tourists,1,1,1,2,2,4,8,8,0,...,1.400984,1.404774,0.587050,0.119323,False,1.5,0.187500,0.312500,2.5,1
2,enticed,1,1,1,3,3,13,7,7,0,...,4.871397,9.543618,3.739086,13.693914,False,3.5,0.500000,0.000000,0.0,2
3,by,1,1,1,4,4,21,2,2,0,...,1.225090,0.755041,0.000363,0.000025,False,NaN,NaN,NaN,NaN,3
4,these,1,1,1,5,5,24,5,5,0,...,2.725317,2.970305,3.010414,1.005914,False,0.5,0.100000,0.400000,2.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515005,countries,20,40,4,6,41,27,9,9,0,...,1.540543,0.629884,0.113662,0.015314,False,0.5,0.055556,0.444444,4.0,515005
515006,are,20,40,4,7,42,37,3,3,0,...,0.839863,0.737262,0.157617,0.021472,False,NaN,NaN,NaN,NaN,515006
515007,pushing,20,40,4,8,43,41,7,7,0,...,6.513069,9.105260,3.740802,5.155717,False,1.5,0.214286,0.285714,2.0,515007
515008,ahead,20,40,4,9,44,49,5,5,0,...,2.104523,1.731191,0.359684,0.064977,False,3.5,0.700000,0.200000,1.0,515008


In [3]:
#Preliminary understanding
# - Text_Nr - Story Identifier(?) - 20 Stories 
# - Trial - 1-40 (Each story is broken down as 40 trials, each of 5 lines (Last trial doesn't necessarily have 5 lines))
# (Presented 5 lines at a time (notes from different paper)) (LINE column has values 1-5)
# Line_WPOS - Word Position in the line
# SERIAL_SCRNUM - Serial Number for word in a given screen (block of upto 5 lines presented at a time, considered as a trial)
# INLET_POSLINE - In a given trial (screen), position of the word in the line, as a measure of characters(?) 
# PP - Participant ID (I think - sa...sj - 10 participants)
# Word_Nr_Trial - Word Number in the trial (but trial here is entiire story and not the 5 line thingy)



In [4]:
dundee_df.columns

Index(['Word', 'Text_Nr', 'Trial', 'LINE', 'LINE_WPOS', 'SERIAL_SCRNUM',
       'INLET_POSLINE', 'OLEN', 'Word_Length', 'PUNCTCODE', 'OMARKS', 'EMARKS',
       'Word_Nr_Trial', 'TXFR', 'IA_Left', 'IA_Right', 'IA_Top', 'IA_Bottom',
       'IA_Area', 'Word_Fixation_Count', 'PP', 'Word_R1_Count',
       'Word_Run_Count', 'endtime', 'starttime', 'Word_R1_Endtime',
       'Word_F1_Starttime', 'Word_F1_X', 'Word_F1_Duration',
       'Word_F1_X_Object', 'Word_F1_X_Word', 'Word_F1_Fixationindex',
       'Word_F1_Visited', 'Word_F1_Progressiveness', 'Word_Gopasttime',
       'Word_R1_Last_X', 'FDUR_y', 'Launch_X', 'Launch_Duration', 'Word_Skip',
       'Word_R1_Gazeduration', 'Trial_Reading_Time', 'Word_Cleaned',
       'Word_Nr_Sentence', 'Sentence_Index', 'Word_Function',
       'Sentence_Length', 'Blinked', 'Word_Cont_or_Func', 'Word_Nr',
       'Word_R1_Starttime', 'Launch_X_all', 'Launch_duration_all',
       'firstfixated', 'startline', 'endline', 'IA_Left_Real', 'IA_Right_Real',
       '

In [29]:
if "pop-os" in platform.node():
    ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/"
else:
    ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT/'
    
TOKENIZER_ROOT = os.path.join(ROOT, "data")
OUT_ROOT = os.path.join(ROOT, "output_dump")
RESULTS_ROOT = os.path.join(ROOT, "results")

SQL_DB = os.path.join(RESULTS_ROOT, "results.db")

def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)

def load_model(model_id, device="cuda", model_path = None):
    """
    Loads a pre-trained GPT model from a checkpoint file.

    Args:
        out_dir (str): The directory where the checkpoint file is located.
        device (torch.device): The device to load the model onto.

    Returns:
        GPT: The loaded GPT model.

    Raises:
        FileNotFoundError: If the checkpoint file is not found.
    """

    if model_path:
        out_dir = model_path
    else:
        conn, c = create_connection_cursor(SQL_DB)
        c.execute("SELECT OutputFolderName FROM Model WHERE ModelID=?", (model_id,))
        out_dir = c.fetchone()[0]
        conn.close()
        
        out_dir = os.path.join(OUT_ROOT, out_dir)
    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    print(f"Loading model from {ckpt_path}")
    # NANOGPT_ROOT = str(Path(__file__).parents[4])

    # Add if condition to check if inside server and if is, then add the path correctly. Default is local for now
    if "pop-os" in platform.node():
        NANOGPT_ROOT = r'/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT'  # Edit later to be dynamic
    else:
        NANOGPT_ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT'
    sys.path.append(NANOGPT_ROOT)
    from model import GPT, GPTConfig

    checkpoint = torch.load(ckpt_path, map_location=device)

    # Backward compatibility for new model args for QKV and FFW Adjustments
    if checkpoint["model_args"].get("wm_decay_length", None) is None:
        # wm_decay_length = block_size
        checkpoint["model_args"]["wm_decay_length"] = checkpoint["model_args"]["block_size"]
    # Setting head size as 3 times n_embd if not set already
    if checkpoint['model_args'].get('head_size_qkv', None) is None:
        checkpoint['model_args']['head_size_qkv'] = checkpoint['model_args']['n_embd']

    if checkpoint["model_args"].get("ffw_dim", None) is None:
        checkpoint["model_args"]["ffw_dim"] = 4 * checkpoint["model_args"]["n_embd"]

    # print(checkpoint['model_args'])
    gptconf = GPTConfig(**checkpoint['model_args'])

    load_model = GPT(gptconf)

    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    load_model.load_state_dict(state_dict)
    load_model.eval()

    load_model = load_model.to(device)

    return load_model

def load_tokenizer(data_dir):
    """
    Load tokenizer for natural stories evaluation.

    Args:
        data_dir (str): The directory path where the tokenizer data is stored.

    Returns:
        tokenizer (Tokenizer): The loaded tokenizer object.

    Raises:
        NotImplementedError: If stoi/itos is not supported or found.

    """
    meta_path = os.path.join(data_dir, 'meta.pkl')
    load_meta = os.path.exists(meta_path)
    if load_meta:
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)
        if meta.get("custom_tokenizer", False):
            print(f"Loading custom tokenizer from {data_dir}")
            tokenizer = AutoTokenizer.from_pretrained(data_dir, use_fast=False)
        else:
            if meta.get("stoi", False):
                raise NotImplementedError("stoi/itos not supported yet")
            else:
                raise NotImplementedError("No stoi/itos found")
    else:
        print("No meta.pkl found, using default GPT-2 tokenizer")
        tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")

    if not tokenizer.eos_token:
        tokenizer.add_special_tokens({"eos_token": "</s>"})
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"  # Add if needed?
    return tokenizer

def load_model_tokenizer(out_dir, data_dir, device="cuda"):
    model = load_model(out_dir, device)
    tokenizer = load_tokenizer(data_dir)
    return model, tokenizer

def return_surprisals(model, context_window_tensor, output_tensor, device='cuda'):
    """
    Given a model. given a context window, give a token, return the surprisal score for the token 
    :param model: 
    :param context_window: (batch_size, context_window) (A list of tokens, needs to be converted to tensor and could be unequal length, so pad with 0 to the left. 0 is "<|endoftext|>" token)
    :param token_list: (batch_size, 1)
    :param device: 
    :return: 
    """
    
    model = model.to(device)
    context_window_tensor = context_window_tensor.to(device)
    output_tensor = torch.tensor(output_tensor).to(device)
    
    #token_tensor = torch.tensor(token_list).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = model(context_window_tensor) #probably don't need the second tensor
    probs = F.log_softmax(logits, dim=-1)
    #print(probs.shape)
    #probs has shape (batch_size, 1, vocab_size)
    #
    #Use the output tensor to get the log probability of the token
    
    token_logprob =  probs.gather(2, output_tensor.unsqueeze(1)).squeeze()
    #print("H20", probs.gather(2, output_tensor.unsqueeze(1)).shape)
    #SANITY CHECK
    #print("H21", token_logprob.shape, token_logprob.squeeze().shape)
    #print("H22", probs[0, 0, output_tensor[0]])
    return -token_logprob




In [6]:
#Extract Stories from the Dundee Data

dundee_byword_story_df = dundee_df[["Word", "Text_Nr", "Word_Nr_Trial", "Word_Function"]].drop_duplicates()
dundee_story_df = dundee_byword_story_df[["Text_Nr", "Word"]].groupby("Text_Nr").apply(lambda x: " ".join(x["Word"].values)).reset_index()

print(dundee_byword_story_df.shape)
print(dundee_story_df.shape)
print(dundee_story_df.head())

dundee_story_df


(51501, 4)
(20, 2)
   Text_Nr                                                  0
0        1  Are tourists enticed by these attractions thre...
1        2  Tony Blair is engaged in low politics on fox-h...
2        3  The decision of the Human Fertility and Embryo...
3        4  Decisions on the next phase of the unwisely na...
4        5  Barrister, war hero, politician. Officially: m...


/tmp/ipykernel_2638697/3882430954.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dundee_story_df = dundee_byword_story_df[["Text_Nr", "Word"]].groupby("Text_Nr").apply(lambda x: " ".join(x["Word"].values)).reset_index()


,Text_Nr,0
0,1,Are tourists enticed by these attractions thre...
1,2,Tony Blair is engaged in low politics on fox-h...
2,3,The decision of the Human Fertility and Embryo...
3,4,Decisions on the next phase of the unwisely na...
4,5,"Barrister, war hero, politician. Officially: m..."
5,6,"If you believe their critics, our noble Lords ..."
6,7,"The case of Ms Susan Wallace, who went down to..."
7,8,As the first phase of the official inquiry int...
8,9,"As ever, market trends are against the way of ..."
9,10,Rather as Basil Fawlty couldn't stop talking a...


In [7]:
#Rename dundee to exisitng DB format - Text_Nr to StoryID, Add CorpusID = 2, Add WordID column that goes from 1 to n for each story

dundee_byword_story_df = dundee_byword_story_df.rename(columns={"Text_Nr": "StoryID", "Word_Nr_Trial": "WordID", "Word_Function": "POSTag"})
dundee_byword_story_df["CorpusID"] = 2

dundee_byword_story_df

,Word,StoryID,WordID,POSTag,CorpusID
0,Are,1,1,VERB,2
1,tourists,1,2,NOUN,2
2,enticed,1,3,VERB,2
3,by,1,4,ADP,2
4,these,1,5,DET,2
...,...,...,...,...,...
51496,countries,20,2449,NOUN,2
51497,are,20,2450,VERB,2
51498,pushing,20,2451,VERB,2
51499,ahead,20,2452,ADV,2


In [37]:
#Things to do 
# Take sample tokenizer and use it to tokenize 1 story

sample_tokenizer = c.execute("SELECT TokenizerID, TokenizerName from Tokenizer WHERE TokenizerID=1").fetchone()
story_df = pd.read_sql_query("SELECT CorpusID, StoryID, GROUP_CONCAT(Word, ' ') as Story FROM Story WHERE CorpusID=2 GROUP BY CorpusID, StoryID", con=conn)

story_df

#Load the tokenizer 

tokenizer = load_tokenizer(os.path.join(TOKENIZER_ROOT, sample_tokenizer[1]))


#Tokenize the story

sample_story = story_df["Story"].iloc[0]
sample_story_tokenized = tokenizer.encode(sample_story)

#print(sample_story_tokenized)

Loading custom tokenizer from 
/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data/babylm_full_bpe_8k

In [8]:
# from tokenizers.normalizers import Lowercase, Strip, StripAccents, NFD
# from tokenizers import normalizers

# normalizer_pipeline = [NFD(), Lowercase(), Strip(), StripAccents()]
# text_normalizer = normalizers.Sequence(normalizer_pipeline)

# def write_tokenized_story_to_db(tokenizer_name, tokenizer):
#     """
#     Given a tokenizer, this function encodes all stories in the "Story" Table in the database and writes it into TokenizedStory Table
    
#     Selects all stories from story table and concatenates all words in the story. Passes this into the tokenizer story by story and writes the tokenized story into the TokenizedStory Table
    
#     Intermediately, has to keep track of word vs token mapping as one word can have multiple tokens. Writes it into the TokenizedStory Table as 1 row for each token (Possibly multiple rows for each word)
        
#     """
    
#     conn, c = create_connection_cursor(SQL_DB)
#     c.execute('SELECT CorpusID, StoryID, group_concat(Word," ") as FullStory FROM Story GROUP BY CorpusID, StoryID HAVING CorpusID=2')
#     stories = c.fetchall()
    
#     c.execute("SELECT TokenizerID FROM Tokenizer WHERE TokenizerName=?", (tokenizer_name,))
#     tokenizer_id = c.fetchone()[0]

#     for story in stories:
#         corpus_id = story[0]
#         story_id = story[1]
#         story_text = story[2]
        
#         #Manually edit one problematic word in the story
#         if story_id == 15:    
#             story_text = story_text.replace("Poilƒne", "Poilâne")
        
#         tokenized_story = tokenizer.encode(story_text)
        
#         c.execute("SELECT StoryWordID, Word FROM Story WHERE CorpusID=? AND StoryID=? ORDER BY WordID", (corpus_id, story_id))
#         words = c.fetchall()
        
#         token_index = 0        
#         for i, word in tqdm.tqdm(enumerate(words)):
#             storyword_id = word[0]
#             decode_list = []
#             if story_id == 15 and "Poilƒne" in word[1]:
#                 print("replacing Poilƒne with Poilâne")
#                 word = (word[0], word[1].replace("Poilƒne", "Poilâne"))

#             while True:
#                 if token_index >= len(tokenized_story):
#                     break
#                 decode_list.append(tokenized_story[token_index])
#                 token_index += 1
#                 if tokenizer.decode(decode_list).strip() == text_normalizer.normalize_str(word[1]):
#                     break
#             for j, token in enumerate(decode_list):
#                 c.execute("INSERT INTO TokenizedStory (TokenizerID, StoryWordID, TokenKey, TokenValue) VALUES (?, ?, ?, ?)", (tokenizer_id, storyword_id, j+1, token))
                
#                 if c.rowcount == 0:
#                     print(f"Error writing tokenized story for {storyword_id}")
#                     break
#     conn.commit()
#     conn.close()

# tokenizer_list = ["babylm_full_bpe_8k", "babylm_full_bpe", "babylm_full_bpe_100M_8k", "babylm_wocdes_full_bpe"]

# for tokenizer_names in tokenizer_list:
#     tokenizer = load_tokenizer(os.path.join(TOKENIZER_ROOT, tokenizer_names))
#     write_tokenized_story_to_db(tokenizer_names, tokenizer)

In [9]:
# c.execute("SELECT StoryWordID, Word FROM Story WHERE CorpusID=? AND StoryID=? ORDER BY WordID", (2, 15))
# for row in c.fetchall():
#     # if row[0] > 48843 and row[0] < 48870:
#     #     print(row)

#     if row[1] == "Poilƒne":
#         print(row)

In [43]:
#Data Quality Check
#For each story for tokenizer ID 1, get the tokens in a list and decode it. Compare it with the original story. If it doesn't match, print the story ID
from tokenizers.normalizers import Lowercase, Strip, StripAccents, NFD
from tokenizers import normalizers

normalizer_pipeline = [NFD(), Lowercase(), Strip(), StripAccents()]
text_normalizer = normalizers.Sequence(normalizer_pipeline)

def calculate_different_words(string1, string2):
    string1_list= string1.split()
    string2_list = string2.split()

    string1_str = "".join(string1_list)
    string2_str = "".join(string2_list)

    if string1_str != string2_str:
        return False
    else:
        return True

token_story_df = pd.read_sql_query("""
WITH TokenizedStoryPart AS(
                SELECT * FROM 
                  TokenizedStory
                  JOIN Story ON TokenizedStory.StoryWordID = Story.StoryWordID
                  WHERE TokenizerID=1 AND CorpusID=2
                  )
SELECT StoryID, group_concat(TokenValue, ',') as TokenizedStory
FROM TokenizedStoryPart
GROUP BY CorpusID, StoryID
           """, con=conn)

token_story_df["TokenizedStory"] = token_story_df["TokenizedStory"].apply(lambda x: [int(i) for i in x.split(",")])
test_tokenizer = load_tokenizer(os.path.join(TOKENIZER_ROOT, "babylm_full_bpe_8k"))

token_story_df["DecodedStory"] = token_story_df["TokenizedStory"].apply(lambda x: test_tokenizer.decode(x))

token_story_df = token_story_df.merge(dundee_story_df, left_on="StoryID", right_on="Text_Nr").drop("Text_Nr", axis=1)
token_story_df["OGStory"] = token_story_df[0].apply(lambda x: text_normalizer.normalize_str(x))
token_story_df.drop(0, axis=1, inplace=True)
token_story_df["Match"] = token_story_df["DecodedStory"] == token_story_df["OGStory"]


token_story_df["DifferentWords"] = token_story_df.apply(lambda x: calculate_different_words(x["DecodedStory"], x["OGStory"]), axis=1)

token_story_df

Loading custom tokenizer from /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data/babylm_full_bpe_8k


,StoryID,TokenizedStory,DecodedStory,OGStory,Match,DifferentWords
0,1,"[435, 2176, 1342, 987, 3616, 344, 628, 3849, 5...",are tourists enticed by these attractions thre...,are tourists enticed by these attractions thre...,False,True
1,2,"[957, 63, 546, 894, 229, 6043, 203, 2102, 5092...",tony blair is engaged in low politics on fox-h...,tony blair is engaged in low politics on fox-h...,False,True
2,3,"[251, 3632, 202, 173, 1790, 187, 2000, 1503, 2...",the decision of the human fertility and embryo...,the decision of the human fertility and embryo...,False,True
3,4,"[576, 41, 3762, 232, 173, 1077, 7582, 202, 173...",decisions on the next phase of the unwisely na...,decisions on the next phase of the unwisely na...,False,True
4,5,"[40, 2178, 1235, 12, 843, 3136, 12, 3650, 14, ...","barrister, war hero, politician. officially: m...","barrister, war hero, politician. officially: m...",True,True
5,6,"[333, 208, 1289, 501, 6693, 12, 595, 7095, 208...","if you believe their critics, our noble lords ...","if you believe their critics, our noble lords ...",False,True
6,7,"[251, 1388, 202, 7975, 5936, 1945, 498, 12, 41...","the case of ms susan wallace, who went down to...","the case of ms susan wallace, who went down to...",False,True
7,8,"[209, 173, 555, 7582, 202, 173, 3203, 6072, 25...",as the first phase of the official inquiry int...,as the first phase of the official inquiry int...,False,True
8,9,"[209, 1030, 12, 2787, 1639, 185, 57, 288, 1171...","as ever, market trends are against the way of ...","as ever, market trends are against the way of ...",False,True
9,10,"[56, 675, 271, 1318, 291, 187, 772, 50, 618, 1...",rather as basil fawlty couldn't stop talking a...,rather as basil fawlty couldn't stop talking a...,False,True


In [9]:


def return_surprisals(model, token_list, device='cuda'):
    if len(token_list)>model.config.block_size:
        token_list = token_list[-model.config.block_size:]
    token_tensor = torch.tensor(token_list).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = model(token_tensor[:, :-1], token_tensor[:, 1:]) #probably don't need the second tensor
    probs = F.log_softmax(logits, dim=-1)
    token_logprob = probs[0, -1, token_tensor[0, -1]].item()
    return -token_logprob

def get_model_surprisals(model, tokenized_story_df, story_id, tokenizer):
    tokenized_story = tokenized_story_df[tokenized_story_df['StoryID'] == story_id].to_dict('records')
    tokenized_story = sorted(tokenized_story, key=lambda x: x['WordID'])
    #Prepend the bos token
    logits_input_list = [tokenizer.bos_token_id]

    for word_row in tqdm(tokenized_story, leave=False):
        tokens = word_row['tokens']
        if len(tokens) == 1:
            logits_input_list.append(tokens[0])
            word_surprisal = return_surprisals(model, logits_input_list)
        else:
            word_surprisal = 0
            for token in tokens:
                logits_input_list.append(token)
                word_surprisal += return_surprisals(model, logits_input_list)

        word_row['surprisal'] = word_surprisal

    return tokenized_story



processing_story_df = pd.read_sql_query("""WITH SubStory as (
SELECT StoryWordID, CorpusID, StoryID, WordID, Word FROM Story
WHERE CorpusID = 2
),
SubTokenStory as (
SELECT * From TokenizedStory Where TokenizedStory.TokenizerID=1
),
JoinedToken as (
SELECT * 
FROM SubStory
Join SubTokenStory
on SubStory.StoryWordID = SubTokenStory.StoryWordID
ORDER BY StoryID, WordID, TokenKey
)
SELECT StoryWordID, StoryID, WordID, Word, group_concat(TokenValue) as tokens
FROM JoinedToken
GROUP BY StoryID, WordID
""", con=conn)


processing_story_df["tokens"] = processing_story_df["tokens"].apply(lambda x: [int(i) for i in x.split(",")])


TEMP_MODEL_PATH = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/output_dump/out-babylm_full_bpe_8k-6x6-mask_ee002_em10-8612957"
temp_model = load_model(0,model_path=TEMP_MODEL_PATH)
temp_tokeizer = load_tokenizer(os.path.join(TOKENIZER_ROOT, "babylm_full_bpe_8k"))


response_df = pd.DataFrame(get_model_surprisals(temp_model, processing_story_df, 1, temp_tokeizer))
print("H2")
response_df

Loading model from /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/output_dump/out-babylm_full_bpe_8k-6x6-mask_ee002_em10-8612957/ckpt.pt
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled


/tmp/ipykernel_405583/3616179155.py:64: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
number of parameters: 13.69M
Loading custom tokenizer from /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data/babylm_full_bpe_8k


H2


,StoryWordID,StoryID,WordID,Word,tokens,surprisal
0,10257,1,1,Are,[435],10.880363
1,10258,1,2,tourists,"[2176, 1342]",12.654627
2,10259,1,3,enticed,"[987, 3616]",18.006316
3,10260,1,4,by,[344],3.913068
4,10261,1,5,these,[628],5.425468
...,...,...,...,...,...,...
2568,12825,1,2569,their,[501],3.585244
2569,12826,1,2570,knees.,"[7272, 14]",7.670453
2570,12827,1,2571,Now,[487],7.031469
2571,12828,1,2572,that's,"[230, 225]",4.299477


In [57]:

def get_context_list(token_string, context_window, tokenizer):

    bos_token = tokenizer.bos_token_id

    context_list = [bos_token]

    if token_string:
        context_list.extend([int(i) for i in token_string.split(",")])
    else:
        return context_list
    
    if len(context_list) > context_window:
        context_list = context_list[-context_window:]

    return context_list


def get_surprisal_inputs(model_id, story_id=None, model_path=None, tokenizer=None):
    conn, c = create_connection_cursor(SQL_DB)

    #For a given model, get its tokenizer id and context window size
    c.execute("SELECT TokenizerID, BlockSize FROM Model WHERE ModelID=?", (model_id,))
    model_row = c.fetchone()
    if model_row is None:
        print("Model ID not found in the database")
        return None
    
    tokenizer_id = model_row[0]
    context_window = model_row[1]
    story_id_part = ""
    if story_id:
        story_id_part = f"AND StoryID={story_id}"


    #Query to get the context window of n or less words before the word given a story id
    query_fin = f"""  WITH    StoryWordRank 
                        AS  (SELECT  Story.StoryWordID,  
                                    Story.StoryID, 
                                    TokenizedStory.TokenValue,
                                    row_number() 
                                        OVER (
                                            PARTITION BY Story.CorpusID, Story.StoryID) AS 
                                    story_token_rank
                            FROM    TokenizedStory
                                    JOIN Story 
                                        On TokenizedStory.StoryWordID = Story.StoryWordID
                            WHERE TokenizedStory.TokenizerID = {tokenizer_id} {story_id_part} AND Story.CorpusID = 2) 
                    SELECT  StoryWordRank.StoryWordID,
                            StoryWordRank.StoryID, 
                            StoryWordRank.TokenValue,
                            
                            group_concat(TokenValue) 
                                OVER (
                                    PARTITION BY StoryID 
                                    ORDER BY story_token_rank ROWS BETWEEN {context_window} PRECEDING AND 1 PRECEDING)  
                                    
                                As 
                                ContextWindow 
                    FROM StoryWordRank """
    
    #c.execute(query_fin)
    #Query returns StoryID, Word, ContextWindow
    #tokenwise_context_story = c.fetchall()
    tokenwise_context_story = pd.read_sql_query(query_fin, con=conn)
    conn.close()

    tokenwise_context_story["ContextWindow"] = tokenwise_context_story["ContextWindow"].apply(lambda x: get_context_list(x, context_window, tokenizer))
    tokenwise_context_story["TokenValue"] = tokenwise_context_story["TokenValue"].apply(lambda x: int(x))

    return tokenwise_context_story

def df_calculate_surprisal(model, context_window, output_token, device='cuda'):
    """
    Given a model, context window and output token, calculate the surprisal for the output token
    """
    context_window_tensor = torch.tensor(context_window).unsqueeze(0).to(device)
    output_tensor = torch.tensor(output_token).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = model(context_window_tensor)

    probs = F.log_softmax(logits, dim=-1)
    token_logprob = probs[0, -1, output_tensor[0, -1]].item()

    return -token_logprob

def df_calculate_surprisal_batch(model, context_window_df, context_window_length, batch_size = 256, device='cuda'):
    """
    Given a model, context window and output token, calculate the surprisal for the output token. Batch data for different length context windows
    """

    for j in tqdm(range(context_window_length)):
        single_context_window = context_window_df[context_window_df["context_window_length"] == j+1]
        context_window = single_context_window["ContextWindow"].tolist()
        output_token = single_context_window["TokenValue"].tolist()
        
        for i in tqdm(range(0, len(context_window), batch_size), leave=False):
            #print("i is", i, "j is", j)
            context_window_tensor = torch.tensor(context_window[i:i+batch_size]).to(device)
            output_tensor = torch.tensor(output_token[i:i+batch_size]).to(device)

            with torch.no_grad():
                logits, _ = model(context_window_tensor)

            probs = F.log_softmax(logits, dim=-1)
            token_logprob = probs.gather(2, output_tensor.unsqueeze(1).unsqueeze(2))

            if j == 0 and i == 0:
                #print("H1")
                surprisal_tensor = -token_logprob
                #print(surprisal_tensor.shape, surprisal_tensor)
            else:
                #print("H2")
                #print(surprisal_tensor.shape, surprisal_tensor, -token_logprob)
                surprisal_tensor = torch.cat((surprisal_tensor, -token_logprob))

    context_window_df["Surprisal"] = surprisal_tensor.squeeze().tolist()

    return context_window_df


    # for i in range(0, len(context_window), batch_size):
    #     context_window_tensor = torch.tensor(context_window[i:i+batch_size]).to(device)
    #     output_tensor = torch.tensor(output_token[i:i+batch_size]).to(device)

    #     #print(context_window_tensor.shape, output_tensor.shape)
    #     with torch.no_grad():
    #         logits, _ = model(context_window_tensor)

    #     probs = F.log_softmax(logits, dim=-1)
    #     #print(probs.shape, logits.shape)
    #     token_logprob = probs.gather(2, output_tensor.unsqueeze(1).unsqueeze(2)).squeeze()
    #     #print(token_logprob.shape)
    #     if i == 0:
    #         surprisal_tensor = -token_logprob
    #     else:
    #         surprisal_tensor = torch.cat((surprisal_tensor, -token_logprob))

    # return surprisal_tensor.tolist()


def process_model_story(model_id):
    loaded_model = load_model(model_id)
    model_tokenizer = c.execute("SELECT TokenizerName FROM Tokenizer WHERE TokenizerID=(SELECT TokenizerID FROM Model WHERE ModelID=?)", (model_id,)).fetchone()
    loaded_tokenizer = load_tokenizer(os.path.join(TOKENIZER_ROOT, model_tokenizer[0]))

    processing_story_df = get_surprisal_inputs(model_id, tokenizer=loaded_tokenizer)
    processing_story_df = processing_story_df.reset_index(drop=True)
    processing_story_df["context_window_length"] = processing_story_df["ContextWindow"].apply(lambda x: len(x))
    processing_story_df["row_id"] = processing_story_df.index
    processing_story_df.sort_values(by=["context_window_length"], inplace=True)
    
    processing_story_df = processing_story_df.reset_index(drop=True)

    processing_story_df = df_calculate_surprisal_batch(loaded_model, processing_story_df, processing_story_df["context_window_length"].max())

    processing_story_df.sort_values(by=["row_id"], inplace=True)
    processing_story_df = processing_story_df.groupby("StoryWordID").agg({"Surprisal": "sum"}).reset_index()
    processing_story_df["ModelID"] = model_id
    processing_story_df = processing_story_df.rename(columns={"Surprisal": "SurprisalScore"})


    return processing_story_df[["ModelID", "StoryWordID", "SurprisalScore"]]


# v2_processing_story_df = get_surprisal_inputs("8612957", tokenizer=temp_tokeizer)
# v2_processing_story_df = v2_processing_story_df.reset_index(drop=True)
# v2_processing_story_df["context_window_length"] = v2_processing_story_df["ContextWindow"].apply(lambda x: len(x))
# v2_processing_story_df["row_id"] = v2_processing_story_df.index
# v2_processing_story_df.sort_values(by=["context_window_length"], inplace=True)

# v2_processing_story_df = v2_processing_story_df.reset_index(drop=True)

# v2_processing_story_df = df_calculate_surprisal_batch(temp_model, v2_processing_story_df, v2_processing_story_df["context_window_length"].max())

# v2_processing_story_df.sort_values(by=["row_id"], inplace=True)

# v2_processing_story_df["Surprisal"] = v2_processing_story_df.progress_apply(lambda x: df_calculate_surprisal(temp_model, x["ContextWindow"], [x["TokenValue"]]), axis=1)


# # v2_processing_story_df["padded_context"] = v2_processing_story_df["ContextWindow"].apply(lambda x: [0]*(256 - len(x)) + x)
# v2_processing_story_df["Surprisal_v2"] = df_calculate_surprisal_batch(temp_model, v2_processing_story_df["ContextWindow"].tolist(), v2_processing_story_df["TokenValue"].tolist())

#v2_processing_story_df


v2_processing_story_df = process_model_story(8612957)

Loading model from /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/output_dump/out-babylm_full_bpe_8k-6x6-mask_ee002_em10-8612957/ckpt.pt
Setting flash to False because wm_mask is enabled


/tmp/ipykernel_405583/3616179155.py:64: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
number of parameters: 13.69M
Loading custom tokenizer from /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data/babylm_full_bpe_8k


100%|██████████| 256/256 [02:36<00:00,  1.63it/s]


In [ ]:
v2_processing_story_df

In [50]:
v2_processing_story_df = v2_processing_story_df.groupby("StoryWordID").agg({"Surprisal": "sum"}).reset_index()

In [ ]:
v2_processing_story_df = v2_processing_story_df.merge(pd.read_sql_query("SELECT StoryWordID, StoryID, WordID, Word FROM Story WHERE CorpusID=2", con=conn), on="StoryWordID").sort_values(by="Surprisal", ascending=False)

v2_processing_story_df

,StoryWordID,Surprisal,Word_x,StoryID,WordID,Word_y
0,25949,79.175491,mˆl‚e,7,453,mˆl‚e
1,27520,67.867405,na‹vet‚.,7,2024,na‹vet‚.
2,32125,67.336423,let's-find-a-cure-fast,9,1384,let's-find-a-cure-fast
3,37279,65.236461,innocent-until-proven-guilty,11,1294,innocent-until-proven-guilty
4,49882,62.749664,twentysomething,16,682,twentysomething
...,...,...,...,...,...,...
51496,45572,0.000750,hour,14,1709,hour
51497,31490,0.000702,from,9,749,from
51498,57327,0.000677,as,19,518,as
51499,43508,0.000671,to,13,2269,to


In [114]:
response_df_joined = response_df.merge(v2_processing_story_df.groupby(["StoryID", "StoryWordID"]).agg({"TokenValue": lambda x: list(x), "Surprisal": "sum", "Surprisal_v2":"sum"}).reset_index(), left_on="StoryWordID", right_on="StoryWordID")

#response_df_joined = response_df.merge(v2_processing_story_df, left_on="StoryWordID", right_on="StoryWordID")

response_df_joined["Surprisal_diff"] = abs(response_df_joined["Surprisal"] - response_df_joined["surprisal"])
response_df_joined["Surprisal_diff_v2"] = abs(response_df_joined["Surprisal_v2"] - response_df_joined["surprisal"])
response_df_joined

,StoryWordID,StoryID_x,WordID,Word,tokens,surprisal,StoryID_y,TokenValue,Surprisal,Surprisal_v2,Surprisal_diff,Surprisal_diff_v2
0,10257,1,1,Are,[435],10.880363,1,[435],10.880363,10.880363,0.000000e+00,0.000000e+00
1,10258,1,2,tourists,"[2176, 1342]",12.654627,1,"[2176, 1342]",12.654626,12.010662,8.940697e-07,6.439649e-01
2,10259,1,3,enticed,"[987, 3616]",18.006316,1,"[987, 3616]",18.006319,18.744581,2.861023e-06,7.382650e-01
3,10260,1,4,by,[344],3.913068,1,[344],3.913070,4.732641,1.668930e-06,8.195724e-01
4,10261,1,5,these,[628],5.425468,1,[628],5.425467,5.568858,1.907349e-06,1.433897e-01
...,...,...,...,...,...,...,...,...,...,...,...,...
2568,12825,1,2569,their,[501],3.585244,1,[501],3.463298,3.585244,1.219463e-01,0.000000e+00
2569,12826,1,2570,knees.,"[7272, 14]",7.670453,1,"[7272, 14]",7.696123,7.670454,2.566957e-02,8.344650e-07
2570,12827,1,2571,Now,[487],7.031469,1,[487],7.074433,7.031469,4.296446e-02,4.768372e-07
2571,12828,1,2572,that's,"[230, 225]",4.299477,1,"[230, 225]",4.431974,4.299476,1.324973e-01,3.576279e-07


In [75]:
data_read_path = '/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/results/rundata.xlsx'

run_details_df = pd.read_excel(data_read_path, sheet_name="Run Details")

tokenizer_name_id_map = pd.read_sql_query("SELECT TokenizerID, TokenizerName FROM Tokenizer", con=conn)


#Columns in database that needs to be mapped 
# CREATE TABLE "Model" (
# 	"ModelID"	INTEGER NOT NULL UNIQUE,
# 	"OutputFolderName"	TEXT NOT NULL,
# 	"TokenizerID"	INTEGER NOT NULL,
# 	"NumLayers"	INTEGER NOT NULL,
# 	"NumHeads"	INTEGER NOT NULL,
# 	"BlockSize"	INTEGER NOT NULL,
# 	"EmbeddingDimension"	INTEGER NOT NULL,
# 	"BatchSize"	INTEGER NOT NULL,
# 	"LearningRate"	REAL NOT NULL,
# 	"Seed"	INTEGER NOT NULL,
# 	"Masking"	BOOLEAN NOT NULL,
# 	"MaskType"	TEXT NOT NULL,
# 	"MaskDecayRate"	REAL NOT NULL,
# 	"EchoicMemory"	INTEGER NOT NULL,
# 	"CurriculumLearning"	BOOLEAN NOT NULL,
# 	"CurriculumType"	TEXT,
# 	"Dataset"	TEXT NOT NULL,
# 	FOREIGN KEY("TokenizerID") REFERENCES "Tokenizer"("TokenizerID"),
# 	PRIMARY KEY("ModelID")
# )

# Index(['run_id', 'output_folder_name', 'n_layer', 'n_head', 'block_size',
#        'n_embd', 'batch_size', 'learning_rate', 'seed', 'masking', 'mask_type',
#        'mask_decay_rate', 'echoic_memory', 'curriculum_learning',
#        'curriculum_type', 'dataset', 'log_exists', 'output_exists',
#        'ckpt_exists', 'sample_exists', 'blimp_exists', 'wandb_exists',
#        'reading_time_exists', 'model_surprisal_data_exists', 'runtime',
#        'rough_sbu_estimate', 'epochs', 'wandb_runid', 'num_iterations',
#        'gradient_accumulation_steps'],
#       dtype='object')


run_details_mapper = {
    "run_id": "ModelID",
    "output_folder_name": "OutputFolderName",
    "n_layer": "NumLayers",
    "n_head": "NumHeads",
    "block_size": "BlockSize",
    "n_embd": "EmbeddingDimension",
    "batch_size": "BatchSize",
    "learning_rate": "LearningRate",
    "seed": "Seed",
    "masking": "Masking",
    "mask_type": "MaskType",
    "mask_decay_rate": "MaskDecayRate",
    "echoic_memory": "EchoicMemory",
    "curriculum_learning": "CurriculumLearning",
    "curriculum_type": "CurriculumType",
    "dataset": "Dataset"
}

model_ids_present = list(pd.read_sql_query("SELECT ModelID FROM Model", con=conn).values.flatten())
model_ids_present.append(6607670)

run_details_df = run_details_df.rename(columns=run_details_mapper)
run_details_df = run_details_df[run_details_mapper.values()]

run_details_df = run_details_df.merge(tokenizer_name_id_map, left_on="Dataset", right_on="TokenizerName", how="left")

run_details_df = run_details_df.drop("TokenizerName", axis=1)

run_details_df = run_details_df[~run_details_df["ModelID"].isin(model_ids_present)]
run_details_df = run_details_df.reset_index(drop=True)



run_details_df


# #Insert into database

# for i, row in run_details_df.iterrows():
#     c.execute("INSERT INTO Model (ModelID, OutputFolderName, TokenizerID, NumLayers, NumHeads, BlockSize, EmbeddingDimension, BatchSize, LearningRate, Seed, Masking, MaskType, MaskDecayRate, EchoicMemory, CurriculumLearning, CurriculumType, Dataset) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", 
#               (row['ModelID'], row['OutputFolderName'], row['TokenizerID'], row['NumLayers'], row['NumHeads'], row['BlockSize'], row['EmbeddingDimension'], row['BatchSize'], row['LearningRate'], row['Seed'], row['Masking'], row['MaskType'], row['MaskDecayRate'], row['EchoicMemory'], row['CurriculumLearning'], row['CurriculumType'], row['Dataset']))
    
    
# conn.commit()

,ModelID,OutputFolderName,NumLayers,NumHeads,BlockSize,EmbeddingDimension,BatchSize,LearningRate,Seed,Masking,MaskType,MaskDecayRate,EchoicMemory,CurriculumLearning,CurriculumType,Dataset,TokenizerID
0,8569444,out-babylm_full_bpe_100M_8k-6x6-nomask-8569444,6,6,256,384,32.0,0.0005,1337,False,Non,0.0,1,False,NaN,babylm_full_bpe_100M_8k,3.0
1,8569446,out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em1...,6,6,256,384,32.0,0.0005,1337,True,exponential_new,2.0,10,False,NaN,babylm_full_bpe_100M_8k,3.0
2,8612955,out-babylm_full_bpe_8k-6x6-nomask-8612955,6,6,256,384,32.0,0.0005,1337,False,Non,0.0,1,False,NaN,babylm_full_bpe_8k,1.0
3,8612957,out-babylm_full_bpe_8k-6x6-mask_ee002_em10-861...,6,6,256,384,32.0,0.0005,1337,True,exponential_new,2.0,10,False,NaN,babylm_full_bpe_8k,1.0


In [77]:
#From Literature - https://pure.rug.nl/ws/portalfiles/portal/216916150/Chapter_2.pdf 
# Eye-movement measures can be further divided into “global” and “local”
# measures. Global measures are aggregated over regions within sentences or
# multiple sentences that form texts. Some typical global measures include mean
# fixation duration (i.e., mean duration of all fixations in a sentence or text) and
# mean saccade length (i.e., mean length of all saccades in a sentence or text). Local
# measures focus on smaller units of text, usually single words. These word-level
# measures can be further divided into "early" measures that reflect rapid processes
# involved in reading, such as lexical access, versus "late" measures that reflect
# subsequent reading processes, such as syntactic integration (Clifton et al., 2007;
# Vasishth et al., 2013).
# 
#  Early measures include first-fixation duration (i.e., the duration of the initial fixation on a word conditional upon it occurring during
# first-pass reading) and 
# Gaze duration (i.e., the sum of all first-pass fixations). 
# Late measures include go-past time (i.e., the sum of all fixations from when the eyes first fixate on the word to when the eyes move to the right off the word, including # all regressions to the left of the word) and 
# total-reading time (i.e., the sum of all fixations on a word, irrespective of whether the fixations occur after a regression).
# These word-level measures are typically used to investigate word-level linguistic
# variables, such as word frequency (i.e., words that occur frequently in text tend
# to be the recipients of fewer, shorter fixations than infrequent words; Schilling et
# al., 1998). This dichotomy is not strict, however, because word-level measures
# have been used to study post-lexical integration (e.g., Warren et al., 2009) and
# other higher-level linguistic variables (e.g., violations of semantic plausibility;
# Rayner et al., 2004; Warren & McConnell, 2007), as well as non-linguistic
# processing (e.g., gender stereotypes; Sturt, 2003).

In [ ]:
#Things to do
# - Ensure all words have unigram frequency and are in the "Word" table alonng with their character length and stuff


#Create table for storing gaze duration (?) along with user level data
#Check other papers to see what they used - Oh and Schuler and Wilcox 



In [ ]:
#Gaze duration - Possible columns - Word R1 GazeDuration(sum of all first-pass fixations (?)), Word F1 Duration (Duration of first fixation), Word_Gopasttime (Sum of all fixations from when the eyes first fixate on the word to when the eyes move to the right off the word, including all regressions to the left of the word)

#Word Fixation Count  + Word Skipped (To udnerstand if words were fixated on or skipped (? ) (Are they inverse?))

In [ ]:
#Select Relevant Columns

# "Word", "Text_Nr", "Word_Nr_Trial","Word_Fixation_Count", "Word_Skipped", "Word_R1_GazeDuration", "Word_F1_Duration", "Word_GoPastTime"


rt_dundee_db_df = dundee_df[['Text_Nr', 'Word_Nr_Trial', "PP", 'Word_Fixation_Count', 'Word_Skip', 'Word_R1_Gazeduration', 'Word_F1_Duration', 'Word_Gopasttime']]


#Rename columns to match the database 
#PP = WorkerID, Text_Nr = StoryID, Word_Nr_Trial = WordID, Word_Fixation_Count = WordFixationCount, Word_Skip = WordSkipped, Word_R1_Gazeduration = GazeDuration, Word_F1_Duration = FirstFixationDuration, Word_Gopasttime = GoPastTime


rt_dundee_db_df = rt_dundee_db_df.rename(columns={"PP": "WorkerID", "Text_Nr": "StoryID",
                                                  "Word_Nr_Trial": "WordID", "Word_Fixation_Count": "WordFixationCount", "Word_Skip": "WordSkipped", "Word_R1_Gazeduration": "GazeDuration", "Word_F1_Duration": "FirstFixationDuration", "Word_Gopasttime": "GoPastTime"})

rt_dundee_db_df = rt_dundee_db_df.merge(pd.read_sql_query("SELECT StoryID, WordID, StoryWordID FROM Story WHERE CorpusID=2", con=conn), on=["StoryID", "WordID"])

#Fill columns GazeDuration, FirstFixationDuration, GoPastTime with 0 if they are NaN

rt_dundee_db_df["GazeDuration"] = rt_dundee_db_df["GazeDuration"].fillna(0)
rt_dundee_db_df["FirstFixationDuration"] = rt_dundee_db_df["FirstFixationDuration"].fillna(0)
rt_dundee_db_df["GoPastTime"] = rt_dundee_db_df["GoPastTime"].fillna(0)

rt_dundee_db_df = rt_dundee_db_df[["WorkerID", "StoryWordID", "GazeDuration", "FirstFixationDuration", "GoPastTime", "WordFixationCount", "WordSkipped"]]
rt_dundee_db_df["RTUID"] = rt_dundee_db_df.index + 1


# #Insert into RTDundeeCorpus

# rt_dundee_db_df.to_sql("RTDundeeCorpus", con=conn, if_exists="append", index=False)


rt_dundee_db_df

,WorkerID,StoryWordID,GazeDuration,FirstFixationDuration,GoPastTime,WordFixationCount,WordSkipped,RTUID
0,sa,10257,216.0,216.0,216.0,1.0,0,1
1,sa,10258,156.0,156.0,156.0,1.0,0,2
2,sa,10259,227.0,227.0,401.0,2.0,0,3
3,sa,10260,0.0,0.0,0.0,0.0,1,4
4,sa,10261,187.0,187.0,525.0,3.0,0,5
...,...,...,...,...,...,...,...,...
515005,sj,61753,290.0,110.0,290.0,2.0,0,515006
515006,sj,61754,0.0,0.0,0.0,0.0,1,515007
515007,sj,61755,336.0,181.0,336.0,2.0,0,515008
515008,sj,61756,254.0,254.0,254.0,1.0,0,515009


In [35]:
#Character Length and Unigram Frequency
#Current WordDetails max ID - 2370 (If issues remove after this number)

#rt_dundee_db_df = dundee_df[['Text_Nr', 'Word_Nr_Trial', "PP", "Word"]]

#Read Story table and for each word, add to the WordDetails table if not already present, and get the WordUID and add to the Story table
from string import punctuation

c.execute("SELECT StoryWordID, CorpusID, StoryID, WordID, Word, WordUID FROM Story WHERE CorpusID=2")


def clean_word(word_str):
    """
    
    """
    #clean the word by removing punctuation, capitalization, special characters (Keep hyphen and apostrophe)
    custom_punctuation = punctuation.replace("'", "").replace("-", "")    
    word_str = word_str.lower()
    word_str = word_str.translate(str.maketrans('', '', custom_punctuation))
    
    if len(word_str) == 0:
        return "<UNK>"
    

    #Borderline case of words that are still just characters that were skipped
    if all([charac in ["-", "'"] for charac in word_str]):
        return "<UNK>"

    if word_str[0] == "'":
        word_str = word_str[1:]
    
    if word_str[-1] == "'":
        word_str = word_str[:-1]

    return word_str
    
word_tracker = []
for row in tqdm(c.fetchall()):
    storywordid, corpusid, storyid, wordid, word, worduid = row

    clean_word_str = clean_word(word)

    c.execute("SELECT WordUID FROM WordDetails WHERE Word=?", (clean_word_str,))
    word_uid_table = c.fetchone()


    if not word_uid_table:
        # c.execute("INSERT INTO WordDetails (Word, CharacterLength) VALUES (?, ?)", (clean_word_str, len(clean_word_str)))
        # conn.commit()
        # c.execute("SELECT WordUID FROM WordDetails WHERE Word=?", (clean_word_str,))
        # worduid = c.fetchone()[0]
        # c.execute("UPDATE Story SET WordUID=? WHERE StoryWordID=?", (worduid, storywordid))
        # conn.commit()
        word_tracker.append({
            "actual_word": word,
            "clean_word": clean_word_str,
            "worduid": worduid,
            "storywordid": storywordid,
            "new_word": True
        })
    else:
        # c.execute("SELECT WordUID FROM WordDetails WHERE Word=?", (clean_word_str,))
        # worduid = c.fetchone()[0]
        # c.execute("UPDATE Story SET WordUID=? WHERE StoryWordID=?", (worduid, storywordid))
        # conn.commit()
        word_tracker.append({
            "actual_word": word,
            "clean_word": clean_word_str,
            "worduid": word_uid_table[0],
            "storywordid": storywordid,
            "new_word": False
        })

word_tracker_df = pd.DataFrame(word_tracker)

word_tracker_df


100%|██████████| 51501/51501 [00:32<00:00, 1569.25it/s]


,actual_word,clean_word,worduid,storywordid,new_word
0,Are,are,286,10257,False
1,tourists,tourists,2371,10258,False
2,enticed,enticed,2372,10259,False
3,by,by,17,10260,False
4,these,these,403,10261,False
...,...,...,...,...,...
51496,countries,countries,5644,61753,False
51497,are,are,286,61754,False
51498,pushing,pushing,9798,61755,False
51499,ahead,ahead,3629,61756,False


In [36]:
# Add wordfrequency to the WordDetails table

subtlex_frequencies_path = "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/eval/reading_time_analysis/SUBTLEXusExcel2007.xlsx"
subtlex_frequencies_df = pd.read_excel(subtlex_frequencies_path, dtype={"Word": str}, keep_default_na=False)

#Make word lowercase
subtlex_frequencies_df["Word"] = subtlex_frequencies_df["Word"].str.lower()
subtlex_frequencies_df["Word"] = subtlex_frequencies_df["Word"].astype(str)

assert subtlex_frequencies_df[subtlex_frequencies_df.Word.duplicated()].shape[0] == 0
assert subtlex_frequencies_df[subtlex_frequencies_df.Word.map(lambda x: any(char in punctuation for char in x))].shape[0] == 0
subtlex_frequencies_dict = subtlex_frequencies_df[['Word', 'Lg10WF']].set_index("Word").to_dict(orient="dict")["Lg10WF"]

subtlex_frequencies_df

# #Check for words with punctuation
# subtlex_frequencies_df[subtlex_frequencies_df["Word"].str.contains("[^a-zA-Z0-9\s]")]


,Word,FREQcount,CDcount,FREQlow,Cdlow,SUBTLWF,Lg10WF,SUBTLCD,Lg10CD
0,the,1501908,8388,1339811,8388,29449.176471,6.176644,100.000000,3.923710
1,to,1156570,8383,1138435,8380,22677.843137,6.063172,99.940391,3.923451
2,a,1041179,8382,976941,8380,20415.274510,6.017526,99.928469,3.923399
3,you,2134713,8381,1595028,8376,41857.117647,6.329340,99.916547,3.923348
4,and,682780,8379,515365,8374,13387.843137,5.834281,99.892704,3.923244
...,...,...,...,...,...,...,...,...,...
74281,zoroastrian,1,1,0,0,0.019608,0.301030,0.011922,0.301030
74282,zoroastrianism,1,1,0,0,0.019608,0.301030,0.011922,0.301030
74283,zugzwang,1,1,1,1,0.019608,0.301030,0.011922,0.301030
74284,zygotes,1,1,1,1,0.019608,0.301030,0.011922,0.301030


In [37]:
worddetails_df = pd.read_sql("SELECT * FROM WordDetails WHERE LogFrequencies IS NULL", con=conn)
worddetails_df["Frequencies"] = worddetails_df["Word"].apply(lambda x: subtlex_frequencies_dict.get(x, 0))
print("Words with no frequency data = ", worddetails_df[worddetails_df["Frequencies"] == 0].shape[0])
worddetails_df

Words with no frequency data =  1521


,WordUID,Word,CharacterLength,LogFrequencies,Frequencies
0,2371,tourists,8,None,2.442480
1,2372,enticed,7,None,0.954243
2,2373,attractions,11,None,1.579784
3,2374,threatening,11,None,2.778151
4,2375,sea-lions,9,None,0.000000
...,...,...,...,...,...
7764,10135,disks,5,None,1.913814
7765,10136,stored,6,None,2.230449
7766,10137,embrace,7,None,2.587711
7767,10138,coincides,9,None,1.146128


In [38]:
# c.executemany("UPDATE WordDetails SET LogFrequencies=? WHERE WordUID=?", worddetails_df[["Frequencies", "WordUID"]].values)
# conn.commit()

In [ ]:
# Questions -
# - Which variable to predict - GazeDuration, FirstFixationDuration, GoPastTime


# - Oh Schuler and Wilcox - Both predict Gaze Duration but Wilcox average across subjects

# - Predictors used   - Oh and Schuler - Word Length, Index of position within sentence, Saccade Length, Whether previous word was fixated
#                     - Wilcox - 


In [10]:
def set_restrictions(df, analysis, maxdist=24, prevfix=False,maxdur=900,mindur=70):
    """
    set restrictions on dataframes for analyses
    input:  df = input dataframe
            analysis = 'skipping' of 'readingtimes'
    optional input: maxdist = value of maximum distance (in char) 
                    prevfix = True sellects only instances where the previous word was fixated (defualt=false)
    output: return adjusted dataframe
    
    -- nb mindur and maxdur are ignored when analysis == 'skipping'
    """
    
    # optional: require previous word to be fixated
    if prevfix == True: df.loc[df['Word_Skip'].shift() == 0, 'BAD'] = True
    if 'Blinked' in df: df=df[df['Blinked']==False]

    # set restrictions for word skipping
    if analysis == 'skipping':
        df = df[(df['BAD'] == False) &
                ((df['Word_F1_Progressiveness'] == 1) | (df['Word_Skip'] == 1)) &
                (df['distowrd'] <= maxdist)]
    # set retrictions for reading times
    elif analysis == 'readingtimes':
        df = df[(df['BAD'] == False) &
                (df['Word_F1_Progressiveness'] == 1) &
                (df['distowrd'] <= maxdist) &
                (df['Word_R1_Gazeduration'] <= maxdur) &
                (df['Word_R1_Gazeduration'] >= mindur)]
    else:
        raise ValueError('analysis not recognised!')

    return(df)


cleaned_dundee_df = dundee_df.copy()    
cleaned_dundee_df = set_restrictions(dundee_df, "readingtimes")
print("Number of rows in cleaned dataset = ", cleaned_dundee_df.shape[0])
print("Number of rows in original dataset = ", dundee_df.shape[0])
cleaned_dundee_df.head()

Number of rows in cleaned dataset =  195795
Number of rows in original dataset =  515010


,Word,Text_Nr,Trial,LINE,LINE_WPOS,SERIAL_SCRNUM,INLET_POSLINE,OLEN,Word_Length,PUNCTCODE,...,parafoveal_entropy,parafoveal_surprise,parafoveal_entropy_contextual,parafoveal_surprise_contextual,BAD,word_land_pos,word_land_frac,word_land_dist,word_land_dist_abs,row_UID
1,tourists,1,1,1,2,2,4,8,8,0,...,1.400984,1.404774,0.587050,0.119323,False,1.5,0.187500,0.312500,2.5,1
2,enticed,1,1,1,3,3,13,7,7,0,...,4.871397,9.543618,3.739086,13.693914,False,3.5,0.500000,0.000000,0.0,2
4,these,1,1,1,5,5,24,5,5,0,...,2.725317,2.970305,3.010414,1.005914,False,0.5,0.100000,0.400000,2.0,4
5,attractions,1,1,1,6,6,30,11,11,0,...,2.050047,4.094616,0.977113,0.355122,False,2.5,0.227273,0.272727,3.0,5
8,very,1,1,1,9,9,60,4,4,0,...,4.494581,3.535600,4.427589,3.350713,False,1.5,0.375000,0.125000,0.5,8


/tmp/ipykernel_3167258/4249564595.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_dundee_df_join["drop_row"] = False


In [21]:

cleaned_dundee_df_join = cleaned_dundee_df[["row_UID"]]
cleaned_dundee_df_join["drop_row"] = False

dundee_df_cleaned = dundee_df.merge(cleaned_dundee_df_join, left_on="row_UID", right_on="row_UID", how="left")
dundee_df_cleaned["drop_row"] = dundee_df_cleaned["drop_row"].fillna(True)
dundee_df_cleaned = dundee_df_cleaned[['Text_Nr', 'Word_Nr_Trial', "PP", "drop_row"]]
dundee_df_cleaned = dundee_df_cleaned.rename(columns={"PP": "WorkerID", "Text_Nr": "StoryID",
                                                  "Word_Nr_Trial": "WordID", 
                                                  "drop_row": "IgnoreRow"})

dundee_df_cleaned = dundee_df_cleaned.merge(pd.read_sql_query("SELECT StoryID, WordID, StoryWordID FROM Story WHERE CorpusID=2", con=conn), on=["StoryID", "WordID"])
dundee_df_cleaned = dundee_df_cleaned[[ "StoryWordID", "WorkerID", "IgnoreRow"]]


dundee_df_cleaned.head()

/tmp/ipykernel_3167258/557965506.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_dundee_df_join["drop_row"] = False
/tmp/ipykernel_3167258/557965506.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dundee_df_cleaned["drop_row"] = dundee_df_cleaned["drop_row"].fillna(True)


,StoryWordID,WorkerID,IgnoreRow
0,10257,sa,True
1,10258,sa,False
2,10259,sa,False
3,10260,sa,True
4,10261,sa,False


In [22]:

dundee_df_cleaned

,StoryWordID,WorkerID,IgnoreRow
0,10257,sa,True
1,10258,sa,False
2,10259,sa,False
3,10260,sa,True
4,10261,sa,False
...,...,...,...
515005,61753,sj,False
515006,61754,sj,True
515007,61755,sj,False
515008,61756,sj,False


In [ ]:
# #Update the database - 

# for row in tqdm(dundee_df_cleaned.to_dict('records')):
#     c.execute("UPDATE RTDundeeCorpus SET IgnoreRow=? WHERE StoryWordID=? AND WorkerID=?", (row["IgnoreRow"], row["StoryWordID"], row["WorkerID"]))

# conn.commit()

100%|██████████| 515010/515010 [00:02<00:00, 255538.40it/s]


In [28]:
conn.close()